# Реализуйте высокочастотную фильтрацию на основе ядра Гаусса
# Реализуйте вычисление свертки с помощью преобразования Фурье
# Реализуйте удаление периодического шума с помощью преобразования Фурье

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
 
img = cv.imread('periodic_noise.jpg', cv.IMREAD_GRAYSCALE)
 
dft = cv.dft(np.float32(img),flags = cv.DFT_COMPLEX_OUTPUT)
dft_shift = np.fft.fftshift(dft)

In [ ]:
dft = cv.dft(np.float32(img), flags=cv.DFT_COMPLEX_OUTPUT)
dft_shift = np.fft.fftshift(dft, axes=(0, 1))

rows, cols = img.shape
crow, ccol = rows // 2, cols // 2
u = np.arange(cols) - ccol
v = np.arange(rows) - crow
U, V = np.meshgrid(u, v)
D2 = U**2 + V**2


In [ ]:
def apply_filter(dft_shift, mask):
    f_ishift = np.fft.ifftshift(dft_shift * mask, axes=(0, 1))
    return cv.idft(f_ishift, flags=cv.DFT_SCALE | cv.DFT_REAL_OUTPUT)

In [ ]:
def gaussian_highpass(sigma):
    mask_hp = 1 - np.exp(-D2 / (2 * sigma**2))
    return np.dstack([mask_hp, mask_hp]).astype(np.float32)

In [ ]:
sigmas = [10, 30, 60]

fig, axs = plt.subplots(len(sigmas), 3, figsize=(15, 5 * len(sigmas)))

for row, sigma in zip(axs, sigmas):
    mask_hp = gaussian_highpass(sigma)
    filtered = dft_shift * mask_hp
    magn = 20 * np.log(cv.magnitude(filtered[:, :, 0], filtered[:, :, 1]) + 1)
    img_hp = apply_filter(dft_shift, mask_hp)
    row[0].imshow(mask_hp[:, :, 0], cmap='gray')
    row[0].set_title(f'mask, sigma={sigma}')
    row[1].imshow(magn, cmap='gray')
    row[1].set_title('spectrum')
    row[2].imshow(img_hp, cmap='gray')
    row[2].set_title('high-pass result')

In [ ]:
def fft_convolve(img, kernel):
    h, w = img.shape
    kh, kw = kernel.shape
    r = cv.getOptimalDFTSize(h + kh - 1)
    c = cv.getOptimalDFTSize(w + kw - 1)

    img_pad = np.zeros((r, c), np.float32)
    img_pad[:h, :w] = img
    kernel_pad = np.zeros((r, c), np.float32)
    kernel_pad[:kh, :kw] = kernel

    dft_img = cv.dft(img_pad, flags=cv.DFT_COMPLEX_OUTPUT)
    dft_kernel = cv.dft(kernel_pad, flags=cv.DFT_COMPLEX_OUTPUT)
    dft_res = cv.mulSpectrums(dft_img, dft_kernel, 0)
    res = cv.idft(dft_res, flags=cv.DFT_SCALE | cv.DFT_REAL_OUTPUT)

    return res[kh // 2:kh // 2 + h, kw // 2:kw // 2 + w]

In [ ]:
kernel = cv.getGaussianKernel(9, 2)
kernel = (kernel @ kernel.T).astype(np.float32)

img_fft_conv = fft_convolve(np.float32(img), kernel)
img_filter2d = cv.filter2D(np.float32(img), -1, np.flip(kernel), borderType=cv.BORDER_CONSTANT)

print(f'max abs diff = {np.abs(img_fft_conv - img_filter2d).max():.4f}')
print(f'MSE = {np.mean((img_fft_conv - img_filter2d)**2):.4f}')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(img, cmap='gray')
axs[0].set_title('original')
axs[1].imshow(img_fft_conv, cmap='gray')
axs[1].set_title('FFT convolution')
axs[2].imshow(img_filter2d, cmap='gray')
axs[2].set_title('cv.filter2D')

In [ ]:
magn = cv.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])
magn_log = np.log(magn + 1)
plt.imshow(20 * magn_log, cmap='gray')

R_dc = 15
k_std = 4
r_notch = 5

local_max = (magn_log == cv.dilate(magn_log, np.ones((15, 15), np.uint8)))
peaks = local_max & (magn_log > magn_log.mean() + k_std * magn_log.std())
peaks[D2 <= R_dc**2] = False
peak_points = np.argwhere(peaks)
print(len(peak_points))

In [ ]:
plt.imshow(20 * magn_log, cmap='gray')
plt.scatter(peak_points[:, 1], peak_points[:, 0], s=60, facecolors='none', edgecolors='r')

notch = np.ones((rows, cols), np.float32)
for y, x in peak_points:
    cv.circle(notch, (int(x), int(y)), r_notch, 0, -1)

mask_notch = np.dstack([notch, notch])
plt.imshow(notch, cmap='gray')

In [ ]:
img_denoised = apply_filter(dft_shift, mask_notch)
plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(img_denoised, cmap='gray')
plt.title('Denoised image'), plt.xticks([]), plt.yticks([])
plt.show()